# Precision, recall and F1

In [ ]:
import warnings
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
warnings.simplefilter("ignore")

## Two spam filters and two cancer models with equal accuracy

In [ ]:
def metrics(tn, fp, fn, tp):
    return dict(accuracy=(tp + tn) / (tn + fp + fn + tp), precision=tp / (tp + fp), recall=tp / (tp + fn))

print("spam A  ", metrics(tn=610, fp=100, fn=100, tp=190))
print("spam B  ", metrics(tn=700, fp=10, fn=190, tp=100))
print("cancer A", metrics(tn=750, fp=90, fn=10, tp=150))
print("cancer B", metrics(tn=800, fp=40, fn=60, tp=100))

spam A   {'accuracy': 0.8, 'precision': 0.6551724137931034, 'recall': 0.6551724137931034}
spam B   {'accuracy': 0.8, 'precision': 0.9090909090909091, 'recall': 0.3448275862068966}
cancer A {'accuracy': 0.9, 'precision': 0.625, 'recall': 0.9375}
cancer B {'accuracy': 0.9, 'precision': 0.7142857142857143, 'recall': 0.625}


## F1: harmonic mean of precision and recall

In [ ]:
for p, r in [(0.8, 0.8), (0.6, 1.0), (0.0, 1.0)]:
    f1 = 0 if p + r == 0 else 2 * p * r / (p + r)
    print(f"precision {p}, recall {r}: mean {(p + r) / 2:.2f}, F1 {f1:.2f}")

## Heart disease

In [ ]:
df = pd.read_csv("data/heart.csv")
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, :-1], df.iloc[:, -1], test_size=0.2, random_state=2)
for name, m in [("logistic regression", make_pipeline(StandardScaler(), LogisticRegression())),
                ("decision tree", DecisionTreeClassifier(random_state=1))]:
    y_pred = m.fit(X_train, y_train).predict(X_test)
    print(f"{name:20s} precision {precision_score(y_test, y_pred):.3f}  recall {recall_score(y_test, y_pred):.3f}  F1 {f1_score(y_test, y_pred):.3f}")
    print("   per class (0, 1): precision", precision_score(y_test, y_pred, average=None).round(3))

## Three classes: per class, macro and weighted

In [ ]:
cm = np.array([[25, 10, 5], [1, 30, 3], [3, 11, 20]])      # rows actual dog/cat/rabbit, columns predicted
y_true = np.repeat(np.repeat(np.arange(3), 3), cm.ravel())
y_pred = np.repeat(np.tile(np.arange(3), 3), cm.ravel())
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=["dog", "cat", "rabbit"], digits=3))

## Ten classes: digits

In [ ]:
X, y = load_digits(return_X_y=True)
a, b, c, d = train_test_split(X, y, test_size=0.2, random_state=1)
p = LogisticRegression(max_iter=10000).fit(a, c).predict(b)
print(classification_report(d, p, digits=3))